# Choosing the Latent Distribution in Flow Matching

A fundamental choice in flow matching is the source distribution from which we sample latent variables. While standard Gaussian noise is the default choice, the geometry and tail behavior of the target distribution may suggest alternative latent distributions.

## Key Question

**Can we improve flow matching by adapting the noise distribution componentwise to match the target geometry?**

In this notebook, we explore this question using Neal's funnel distribution, which exhibits different tail behaviors in each dimension. We'll compare three types of latent distributions:

1. **Uniform:** Bounded support, no tails
2. **Gaussian:** Light tails, standard choice
3. **Student-t:** Heavy tails, with componentwise adaptation



## Setup for Google Colab

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print("Installing dependencies...")
    !pip install -q pot geomloss
    
    # Download fm_utils.py
    import urllib.request
    url = "https://raw.githubusercontent.com/USERNAME/REPO/main/fm_utils.py"
    urllib.request.urlretrieve(url, "fm_utils.py")
    print("✓ Setup complete!")
else:
    # Local development - add parent directory to path
    import sys
    sys.path.insert(0, '..')

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from geomloss import SamplesLoss

from fm_utils import (
    DEVICE,
    FlowConfig,
    compute_ema,
    draw_samples,
    euler_integrate,
    get_distribution,
    make_batch_sampler,
    make_latent_sampler,
    plot_funnel_with_marginals,
    plot_latent_colored_by_endpoint_norm,
    set_seed,
    train_flow,
)

# Define distance metric for comparing distributions
mmd_loss = SamplesLoss("energy")

# Configuration: Number of samples for visualization
n_samples = 500_000  # Number of samples for visualization plots

print(f"Using device: {DEVICE}")
print(f"Visualization samples: {n_samples:,}")

## Understanding Neal's Funnel

Neal's funnel is a hierarchical distribution with asymmetric tail behavior:

$$
\begin{align}
x_1 &\sim \mathcal{N}(0, \sigma_1^2) \\
x_2 | x_1 &\sim \mathcal{N}(0, e^{\alpha x_1})
\end{align}
$$

This creates:
- **Dimension 1 ($x_1$)**: Light-tailed Gaussian
- **Dimension 2 ($x_2$)**: Heavy-tailed (variance grows exponentially with $x_1$)

The key insight: **different dimensions have different tail behaviors**, suggesting that we should use different noise distributions per dimension.

### Z-Score Normalization for Stability

The funnel has extreme scale variations that can cause numerical instability. To handle this, we use **z-score normalization**:

$$\text{normalized}_i = \frac{x_i - \mu_i}{\sigma_i}$$

For the funnel:
- **$x_1$**: Mean 0, std $\sigma_1$ (e.g., 3.0)
- **$x_2$**: Mean 0, std $\approx \exp(0.25 \times \alpha^2 \times \sigma_1^2)$

The `ZScoreWrapper` in `fm_utils` applies this automatically, ensuring stable training while preserving the funnel's geometry.

**Why this matters:** Without normalization, $x_2$ can range from -1000 to +1000, making training very difficult.

In [ ]:
set_seed(42)

# Create Neal's funnel distribution
funnel = get_distribution("funnel", sigma1=3.0, alpha=1.0)
funnel_sampler = make_batch_sampler(funnel)
samples = draw_samples(funnel, n=n_samples)

# Visualize the funnel with marginals
fig = plot_funnel_with_marginals(
    samples, 
    funnel,
    title="Neal's Funnel Distribution"
)
plt.show()
plt.close(fig)

print("Notice: $x_2$ has much heavier tails than $x_1$")

## Training Configuration

We'll use the same training setup for all experiments to ensure fair comparison.

By default, we use random pairing between latent and target samples. You can optionally enable `pairing='minibatch_ot'` for structured optimal transport pairings.

In [ ]:
# Base training configuration
base_config = {
    'target_sampler': funnel_sampler,
    'latent_sampler': None,  # Will set per experiment
    'pairing': 'none',
    'steps': 5_000,
    'batch_size': 128,
    'log_every': 1000,
    'lr': 2e-4,
    'seed': 42,
    'flow_T': 1.0,
    'ema_decay': 0.99
}


---

## Exploration 1: Uniform Latents

Let's start with **uniform distributions** which have bounded support and no tails. We use IID uniform with range [-2, 2] as our baseline.

In [ ]:
# Choose your latent sampler here:
latent_sampler_uniform = make_latent_sampler("uniform", DEVICE, 2)

# Visualize the latent distribution
latents = latent_sampler_uniform((n_samples, 2))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(latents.cpu()[:, 0], latents.cpu()[:, 1], s=3, alpha=0.4, color='#3498DB', linewidths=0)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('Uniform Latent Distribution')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Training with Uniform latents...")
print("=" * 60)

config_uniform = base_config.copy()
config_uniform['latent_sampler'] = latent_sampler_uniform

run_uniform = train_flow(FlowConfig(**config_uniform))

In [ ]:

# Generate samples
latents = latent_sampler_uniform((n_samples, 2))
with torch.no_grad():
    generated_uniform = euler_integrate(run_uniform.ema, latents, run_uniform.config.flow_T, steps=150)

# Advanced funnel plot with marginals
fig = plot_funnel_with_marginals(
    generated_uniform, 
    funnel,
    title="Uniform Latents: Generated vs True Distribution"
)
plt.show()
plt.close(fig)

# Latent space analysis
fig = plot_latent_colored_by_endpoint_norm(
    latents,
    generated_uniform,
    title="Uniform: Latent colored by ||x||",
    cmap='plasma'
)
plt.show()
plt.close(fig)

In [ ]:
# Compute Energy MMD distance
target_samples = funnel_sampler(5000, device=DEVICE, dtype=torch.float32)
with torch.no_grad():
    mmd_distance = mmd_loss(generated_uniform[:5000], target_samples).item()

print("Uniform Latents - Energy MMD:")
print("=" * 40)
print(f"Energy MMD: {mmd_distance:.6f}")
print("(Lower is better - indicates closer match to target)")

---

## Exploration 2: Gaussian Latents

Now let's try **Gaussian distributions** which have light tails. We use standard IID Gaussian (mean=0, std=1) - the most common choice in flow matching.

In [ ]:
# Choose your latent sampler here:
latent_sampler_gaussian = make_latent_sampler("gaussian", DEVICE, 2)

# Visualize the latent distribution
latents = latent_sampler_gaussian((n_samples, 2))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(latents.cpu()[:, 0], latents.cpu()[:, 1], s=3, alpha=0.4, color='#3498DB', linewidths=0)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('Gaussian Latent Distribution')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Training with Gaussian latents...")
print("=" * 60)

config_gaussian = base_config.copy()
config_gaussian['latent_sampler'] = latent_sampler_gaussian

run_gaussian = train_flow(FlowConfig(**config_gaussian))

In [ ]:


# Generate samples
latents = latent_sampler_gaussian((n_samples, 2))
with torch.no_grad():
    generated_gaussian = euler_integrate(run_gaussian.ema, latents, run_gaussian.config.flow_T, steps=150)

# Advanced funnel plot with marginals
fig = plot_funnel_with_marginals(
    generated_gaussian, 
    funnel,
    title="Gaussian Latents: Generated vs True Distribution"
)
plt.show()
plt.close(fig)

# Latent space analysis
fig = plot_latent_colored_by_endpoint_norm(
    latents,
    generated_gaussian,
    title="Gaussian: Latent colored by ||x||",
    cmap='plasma'
)
plt.show()
plt.close(fig)

In [ ]:
# Compute Energy MMD distance
with torch.no_grad():
    mmd_distance = mmd_loss(generated_gaussian[:5000], target_samples).item()

print("Gaussian Latents - Energy MMD:")
print("=" * 40)
print(f"Energy MMD: {mmd_distance:.6f}")

---

## Exploration 3: Student-t Latents (Componentwise Adaptation)

### Mathematical Definition

The **Student-t distribution** with degrees of freedom $\nu$ and scale $\sigma$ has probability density:

$$p(x) \propto \left(1 + \frac{x^2}{\nu \sigma^2}\right)^{-\frac{\nu+1}{2}}$$

The **degrees of freedom** $\nu$ controls tail behavior:
- $\nu \to \infty$: Converges to Gaussian $\mathcal{N}(0, \sigma^2)$ (light tails)
- Small $\nu$ (e.g., $\nu=4$): Heavy tails with $\mathbb{P}(|X| > t) \sim t^{-\nu}$

**Intuition:** Lower degrees of freedom allocate more probability mass to extreme values, making rare events more likely.

### Componentwise Adaptation for Neal's Funnel

Neal's funnel has asymmetric tail behavior: $x_1$ is light-tailed while $x_2$ is heavy-tailed. We adapt the latent distribution componentwise:

- **$z_1$**: Student-t with $\nu=20, \sigma=1$ (nearly Gaussian)
- **$z_2$**: Student-t with $\nu=4, \sigma=1$ (heavy tails)

These parameters are from [Pandey et al., 2024](https://arxiv.org/abs/2410.14171).

In [ ]:
# Feel free to experiment with different df values!

latent_sampler_studentt = make_latent_sampler([
    ("student_t", {"df": 20.0, "scale": 1}),  # lighter tails for x1
    ("student_t", {"df": 4.0, "scale": 1})    # heavier tails for x2
], DEVICE, 2)

# Visualize the latent distribution
latents = latent_sampler_studentt((n_samples, 2))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(latents.cpu()[:, 0], latents.cpu()[:, 1], s=3, alpha=0.4, color='#3498DB', linewidths=0)
ax.set_xlabel('$z_1$ (df=20, light tails)')
ax.set_ylabel('$z_2$ (df=4, heavy tails)')
ax.set_title('Student-t Latent Distribution (Non-IID)')
ax.set_aspect('equal','box')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: Asymmetric tail behavior matches the funnel!")

In [ ]:
print("Training with Student-t latents (componentwise adaptation)...")
print("=" * 60)

config_studentt = base_config.copy()
config_studentt['latent_sampler'] = latent_sampler_studentt

run_studentt = train_flow(FlowConfig(**config_studentt))

In [ ]:

# Generate samples
latents = latent_sampler_studentt((n_samples, 2))
with torch.no_grad():
    generated_studentt = euler_integrate(run_studentt.ema, latents, run_studentt.config.flow_T, steps=150)

# Advanced funnel plot with marginals
fig = plot_funnel_with_marginals(
    generated_studentt, 
    funnel,
    title="Student-t (Non-IID): Generated vs True Distribution"
)
plt.show()
plt.close(fig)

# Latent space analysis
fig = plot_latent_colored_by_endpoint_norm(
    latents,
    generated_studentt,
    title="Student-t: Latent colored by ||x||",
    cmap='plasma'
)
plt.show()
plt.close(fig)

In [ ]:
# Compute Energy MMD distance
with torch.no_grad():
    mmd_distance = mmd_loss(generated_studentt[:5000], target_samples).item()

print("Student-t Latents (Non-IID) - Energy MMD:")
print("=" * 40)
print(f"Energy MMD: {mmd_distance:.6f}")

---

## Comparison: Does Componentwise Adaptation Help?

Let's compare all three approaches to see which one works best for the funnel distribution.

In [ ]:
# Training Loss Comparison
fig, ax = plt.subplots(figsize=(9, 4))

# Uniform
ax.plot(run_uniform.losses, color='#E74C3C', alpha=0.2, linewidth=0.5)
ax.plot(compute_ema(run_uniform.losses), label="Uniform", color='#E74C3C', linewidth=2)

# Gaussian (baseline)
ax.plot(run_gaussian.losses, color='#3498DB', alpha=0.2, linewidth=0.5)
ax.plot(compute_ema(run_gaussian.losses), label="Gaussian (baseline)", color='#3498DB', linewidth=2)

# Student-t
ax.plot(run_studentt.losses, color='#2ECC71', alpha=0.2, linewidth=0.5)
ax.plot(compute_ema(run_studentt.losses), label="Student-t (Non-IID)", color='#2ECC71', linewidth=2)

ax.set_xlabel('Training step')
ax.set_ylabel('MSE loss')
ax.set_title('Training Loss Comparison: All Three Latent Distributions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("VISUAL COMPARISON: Generated Samples Quality")
print("="*70)
print("\nComparing all three latent distribution choices...")
print("Focus on the tail behavior in the marginal plots (especially x2)")
print("="*70)

In [ ]:
# Compare all three approaches with detailed funnel plots
for name, generated, color in [
    ("Uniform Latents", generated_uniform, '#E74C3C'),
    ("Gaussian Latents", generated_gaussian, '#3498DB'),
    ("Student-t (Non-IID) ✓", generated_studentt, '#2ECC71')
]:
    print(f"\n{name}")
    print("-" * 50)
    fig = plot_funnel_with_marginals(
        generated,
        funnel,
        title=f"{name}: Generated vs True Distribution"
    )
    plt.show()
    plt.close(fig)

print("\n" + "="*70)
print("Key Observations:")
print("="*70)
print("1. Uniform: Struggles with heavy tails (bounded support)")
print("2. Gaussian: Better, but still misses extreme tail samples")
print("3. Student-t (Non-IID): Best tail match - captures tail behavior correctly!")
print("="*70)

## Key Findings

### Main Results

1. **Bounded distributions struggle:** Uniform latents (bounded support) perform poorly on heavy-tailed targets

2. **IID assumptions are limiting:** Using the same distribution for all dimensions is suboptimal when target geometry varies across dimensions


### Practical Takeaway

When designing flow matching models:
- **Analyze your target distribution's geometry**
- **Consider matching the tail behavior componentwise**

### Learn More

This notebook demonstrates some of the motivation for the paper:

**[Adapting Noise to Data: Generative Flows from 1D Processes](https://arxiv.org/abs/2510.12636)**
